In [2]:
# Mount Google Drive
from google.colab import drive  # access Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# pandas for data handling
import pandas as pd

# path to your dataset
path = "/content/drive/MyDrive/BigData/assignment3/ml-latest-small/"

# load ratings and movies
ratings = pd.read_csv(path + "ratings.csv")
movies = pd.read_csv(path + "movies.csv")

# check data
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
# create mappings for user and movie IDs
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user_map = {id:i for i,id in enumerate(user_ids)}
movie_map = {id:i for i,id in enumerate(movie_ids)}

# apply mapping
ratings["user"] = ratings["userId"].map(user_map)
ratings["movie"] = ratings["movieId"].map(movie_map)

# number of users and movies
num_users = len(user_ids)
num_movies = len(movie_ids)

print("Users:", num_users)
print("Movies:", num_movies)

Users: 610
Movies: 9724


In [5]:
# tensorflow for neural network
import tensorflow as tf

# keras layers
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate

# model API
from tensorflow.keras.models import Model

# inputs
user_input = Input(shape=(1,))
movie_input = Input(shape=(1,))

# embeddings
user_emb = Embedding(num_users, 50)(user_input)
movie_emb = Embedding(num_movies, 50)(movie_input)

# flatten
user_vec = Flatten()(user_emb)
movie_vec = Flatten()(movie_emb)

# combine
x = Concatenate()([user_vec, movie_vec])

# dense layers
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)

# output layer
output = Dense(1)(x)

# build model
model = Model([user_input, movie_input], output)

# compile
model.compile(loss='mse', optimizer='adam')

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 50)     │     30,500 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 50)     │    486,200 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 100)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     12,928 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         65 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 537,949 (2.05 MB)

 Trainable params: 537,949 (2.05 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.fit(
    [ratings["user"], ratings["movie"]],
    ratings["rating"],
    epochs=5,
    batch_size=256,
    verbose=1
)

Epoch 1/5
394/394 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 1.7321
Epoch 2/5
394/394 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.7175
Epoch 3/5
394/394 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 0.6744
Epoch 4/5
394/394 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.6402
Epoch 5/5
394/394 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.6057


In [7]:
import numpy as np

# reverse mapping
inv_movie_map = {v:k for k,v in movie_map.items()}

def recommend_nn(user_id, top_k=10):

    user_idx = user_map[user_id]

    movie_indices = np.arange(num_movies)
    user_array = np.full(num_movies, user_idx)

    # predict ratings
    preds = model.predict([user_array, movie_indices], verbose=0)

    # get top-k
    top_indices = preds.flatten().argsort()[-top_k:][::-1]

    movie_ids = [inv_movie_map[i] for i in top_indices]

    return movie_ids

In [8]:
def show_recommendations(user_id, top_k=10):

    recs = recommend_nn(user_id, top_k)

    rec_movies = movies[movies["movieId"].isin(recs)]

    print(f"\nTop {top_k} Recommendations for User {user_id}:\n")
    print(rec_movies[["movieId", "title"]])

In [9]:
show_recommendations(1, top_k=10)


Top 10 Recommendations for User 1:

      movieId                                              title
2582     3451                Guess Who's Coming to Dinner (1967)
4396     6460                     Trial, The (Procès, Le) (1962)
4782     7121                                  Adam's Rib (1949)
5640    27397  Joint Security Area (Gongdong gyeongbi guyeok ...
6913    64197                                      Hunger (2008)
8019    97866                               Imposter, The (2012)
8366   109241  On the Other Side of the Tracks (De l'autre cô...
8933   136341           Scooby-Doo! and the Samurai Sword (2009)
9083   143031                                    Jump In! (2007)
9497   170705                            Band of Brothers (2001)


In [10]:
def evaluate_user_nn(user_id, k=10):

    recs = set(recommend_nn(user_id, k))

    # relevant movies (rating >= 4)
    relevant = set(
        ratings[
            (ratings["userId"] == user_id) & (ratings["rating"] >= 4)
        ]["movieId"]
    )

    intersection = recs.intersection(relevant)

    precision = len(intersection) / k
    recall = len(intersection) / len(relevant) if len(relevant) > 0 else 0

    print(f"\nUser {user_id}")
    print("Intersection:", list(intersection))
    print(f"Precision@{k}: {precision:.3f}")
    print(f"Recall@{k}: {recall:.3f}")

In [12]:
evaluate_user_nn(1, k=50)


User 1
Intersection: []
Precision@50: 0.000
Recall@50: 0.000
